
# AI Research Paper Summarizer

This notebook fine-tunes **Pegasus** for scientific/academic summarization (ArXiv subset) using **PyTorch** + **Hugging Face** libraries, evaluates with **ROUGE**, **BLEU**, **BERTScore**, and demonstrates inference + saving.


## Installing the Necessary Libraries

In [1]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0
!pip install -q evaluate==0.4.2 rouge-score==0.1.2 bert-score==0.3.13 sentencepiece==0.2.0 sacremoses==0.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.4 MB/s eta 0:00:0000:0100:010m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.6.1 which is incompatible.
bigframes 2.8.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.8.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible re

## Importing Libraries

In [2]:
import os, random, numpy as np, torch
from typing import List
from datasets import load_dataset, DatasetDict
from transformers import (
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAME = "google/pegasus-arxiv"
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 160

OUTPUT_DIR = "./pegasus_arxiv_pytorch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tokenizer = PegasusTokenizer.from_pretrained(MODEL_NAME)
model = PegasusForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)


2025-08-25 07:49:09.078659: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756108149.268396      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756108149.327124      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Device: cuda


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-arxiv and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

## Load Dataset

In [3]:
raw = load_dataset("ccdv/arxiv-summarization")
print(raw)

# Adjust sample sizes for faster runs; increase for better results
train_samples = 4000
val_samples   = 800
test_samples  = 800

train_ds = raw["train"].select(range(min(train_samples, len(raw["train"]))))
val_ds   = raw["validation"].select(range(min(val_samples, len(raw["validation"]))))
test_ds  = raw["test"].select(range(min(test_samples, len(raw["test"]))))
dataset  = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)
dataset


Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})


DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 800
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 800
    })
})

## Preprocessing / Tokenization

In [4]:
def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["article"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["abstract"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length",
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess_function, batched=True, remove_columns=["article", "abstract"])
tokenized


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
})

In [5]:
pip install evaluate rouge-score sacrebleu bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sacrebleu]
Note: you may need to restart the kernel to use updated packages.


In [6]:
!pip install evaluate sacrebleu bert-score rouge-score

## Direct Metric Imports (no evaluate.load)

In [7]:
from rouge_score import rouge_scorer
import sacrebleu
import bert_score

import numpy as np
from typing import List
from transformers import DataCollatorForSeq2Seq

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Postprocess
def postprocess_text(preds: List[str], labels: List[str]):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

# Compute Metrics
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # ✅ ROUGE via rouge-score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = {k: np.mean([rouge.score(l, p)[k].fmeasure for l, p in zip(decoded_labels, decoded_preds)]) 
                    for k in ["rouge1", "rouge2", "rougeL"]}

    # ✅ SacreBLEU
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])

    # ✅ BERTScore
    P, R, F1 = bert_score.score(decoded_preds, decoded_labels, lang="en", verbose=False)

    metrics = {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "sacrebleu": bleu.score,
        "bertscore_precision": float(P.mean()),
        "bertscore_recall": float(R.mean()),
        "bertscore_f1": float(F1.mean()),
    }
    return metrics



## Training Arguments & Trainer

In [8]:
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=0.3,
    warmup_steps=300,
    lr_scheduler_type="linear",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## Fine-tune

In [9]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
train_result.metrics


Step,Training Loss,Validation Loss


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 256, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 256, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}


{'train_runtime': 582.6802,
 'train_samples_per_second': 2.059,
 'train_steps_per_second': 0.257,
 'total_flos': 3467357297049600.0,
 'train_loss': 2.956926066080729,
 'epoch': 0.3}

## Evaluation (Validation & Test)

In [10]:
print("Evaluating on validation set ...")
val_metrics = trainer.evaluate(eval_dataset=tokenized["validation"], max_length=MAX_TARGET_LENGTH, num_beams=4)
print(val_metrics)

print("Evaluating on test set ...")
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"], max_length=MAX_TARGET_LENGTH, num_beams=4, metric_key_prefix="test")
print(test_metrics)


Evaluating on validation set ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 2.960759162902832, 'eval_rouge1': 0.44791454739955283, 'eval_rouge2': 0.16911351174459152, 'eval_rougeL': 0.2652242924847962, 'eval_sacrebleu': 14.163344925004608, 'eval_bertscore_precision': 0.8640501499176025, 'eval_bertscore_recall': 0.8608224391937256, 'eval_bertscore_f1': 0.8623217940330505, 'eval_runtime': 3460.8645, 'eval_samples_per_second': 0.231, 'eval_steps_per_second': 0.231, 'epoch': 0.3}
Evaluating on test set ...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'test_loss': 2.814242362976074, 'test_rouge1': 0.45048828725929574, 'test_rouge2': 0.17306363777848652, 'test_rougeL': 0.26814856429608575, 'test_sacrebleu': 14.112767094079684, 'test_bertscore_precision': 0.8654054403305054, 'test_bertscore_recall': 0.8608667254447937, 'test_bertscore_f1': 0.8630149960517883, 'test_runtime': 3402.2283, 'test_samples_per_second': 0.235, 'test_steps_per_second': 0.235, 'epoch': 0.3}


## Inference Demo

In [11]:
sample_text = """
Introduction: Deep learning has revolutionized NLP, enabling significant advances in sequence modeling.
Methodology: We leverage Pegasus, a transformer encoder-decoder, fine-tuned on scientific corpora to produce section-specific summaries.
Results: The model achieves strong ROUGE and BERTScore on validation data, indicating high overlap and semantic adequacy.
Conclusion: Domain-aware fine-tuning with controlled decoding yields concise, faithful research summaries suited for academic workflows.
"""

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_INPUT_LENGTH).to(device)
gen_ids = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=5, early_stopping=True)
summary = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print("Generated Summary:\n", summary)


Generated Summary:
 in this paper we study the problem of learning a sequence of random numbers from a sequence of random numbers. <n> we show that learning a sequence of random numbers from a sequence of random numbers is equivalent to learning a sequence of random numbers from a sequence of random numbers. <n> we show that learning a sequence of random numbers from a sequence of random numbers is equivalent to learning a sequence of random numbers from a sequence of random numbers. <n> we also show that learning a sequence of random numbers from a sequence of random numbers is equivalent to learning a sequence of random numbers from a sequence of random numbers. <n> the problem of learning a sequence of random numbers from a sequence of random numbers has attracted a lot of attention in recent years @xcite. <n>


## Save & Reload

In [12]:
save_path = os.path.join(OUTPUT_DIR, "final")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
re_tok = AutoTokenizer.from_pretrained(save_path)
re_mod = AutoModelForSeq2SeqLM.from_pretrained(save_path).to(device)

check_ids = re_mod.generate(**tokenizer("This paper studies...", return_tensors="pt", truncation=True, padding=True, max_length=MAX_INPUT_LENGTH).to(device),
                            max_length=MAX_TARGET_LENGTH, num_beams=4)
print("Reloaded Model Summary:\n", re_tok.decode(check_ids[0], skip_special_tokens=True))


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 256, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}


Reloaded Model Summary:
 we present a new method to determine whether or not a given quantum state is a superposition of two superpositions of classical states. <n> the method is based on the observation that the state of a quantum system can be represented by a sum of superpositions of classical states. <n> the state of a quantum system can be represented by a sum of superpositions of classical states. <n> the method can be used to determine whether or not a given quantum state is a superposition of two superpositions of classical states. <n> the quantum state of a system can be represented by a sum of superpositions of classical states. <n> the quantum state of a system can be represented by a sum of superpositions of classical states.
